# Sub-floor probe analysis — Colab runner

Runs the full analysis from `analysis.py`. Every stage checkpoints to
`output/results.json`, so if Colab disconnects you re-run the same cell and it
picks up where it stopped.

**On the T4:** the GPU will almost certainly not help here and may hurt. After
feature selection each fit sees ~250 rows x 500 columns, which is far below the
size where GPU histogram building beats CPU — the per-tree kernel launch overhead
dominates. Cell 5 benchmarks both so you can see the actual numbers rather than
taking my word for it. What *does* help on Colab is that the session survives
longer than a few minutes idle, and `--perm-jobs` uses every core the VM gives you.

Runtime → Change runtime type → **CPU** is a perfectly good choice for this. Pick
the T4 only if you want to run the benchmark, or if a high-RAM GPU runtime happens
to come with more vCPUs than the CPU runtime.

## 1. What machine did we get

In [ ]:
import os, multiprocessing, subprocess
print('vCPUs:', multiprocessing.cpu_count())
print(subprocess.run(['free','-g'], capture_output=True, text=True).stdout)
try:
    print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total',
                          '--format=csv'], capture_output=True, text=True).stdout)
except FileNotFoundError:
    print('no GPU on this runtime (fine — see the note above)')

## 2. Dependencies

In [ ]:
!pip -q install "xgboost>=2.1" "scikit-learn>=1.3" "scipy>=1.11" "pandas>=2.0" joblib
import xgboost, sklearn, scipy, pandas
print('xgboost', xgboost.__version__, '| sklearn', sklearn.__version__,
      '| scipy', scipy.__version__, '| pandas', pandas.__version__)

## 3. Get the code

Public repo — no token needed to read. If you have already cloned it, this pulls
the latest instead of failing.

In [ ]:
REPO = 'https://github.com/VictoryWizard/alz-geo-pipeline.git'
import os
if not os.path.isdir('/content/alz-geo-pipeline'):
    !git clone -q $REPO /content/alz-geo-pipeline
else:
    !cd /content/alz-geo-pipeline && git pull -q
%cd /content/alz-geo-pipeline
!ls

## 4. Get the data

The two series matrices are ~60 MB each and are **not** in the repo (they are
gitignored — never commit them). Pulled straight from GEO.

In [ ]:
import os
os.makedirs('data', exist_ok=True)
BASE = 'https://ftp.ncbi.nlm.nih.gov/geo/series/GSE63nnn'
for g in ['GSE63060', 'GSE63061']:
    f = f'data/{g}_series_matrix.txt.gz'
    if not os.path.exists(f):
        !wget -q -O $f $BASE/$g/matrix/{g}_series_matrix.txt.gz
    print(f, os.path.getsize(f) // 1024 // 1024, 'MB')

## 5. Benchmark: is the T4 actually faster here?

Times one representative fit on each device. Skip if you are on a CPU runtime.

In [ ]:
import time, numpy as np, analysis as A
D, info = A.load_data()
Xa, ya = D['GSE63060']['X'], D['GSE63060']['y']
Xb, yb = D['GSE63061']['X'], D['GSE63061']['y']
T = A.floor_of(Xa)
P1, P2, n = A.prep(Xa, Xb, 'STRICT-100', T)
print('feature matrix handed to the model:', P1.shape, '->', A.CONFIG['N_TOP'], 'after selection')

for dev in ['cpu', 'cuda']:
    try:
        A.DEVICE = dev
        A.fit_score(P1, ya, P2)                      # warm up
        t = time.time()
        for _ in range(5):
            A.fit_score(P1, ya, P2)
        print(f'{dev:>5}: {(time.time()-t)/5:.3f} s per fit')
    except Exception as e:
        print(f'{dev:>5}: unavailable ({type(e).__name__})')
A.DEVICE = 'cpu'

## 6. Fast stages

A1 transfer, A2 within-cohort CV, A3 demographics baseline, A4 flag sensitivity,
A5 model family, R1 seed repeats, R2 floor sweep, R3 KDE stability.
Roughly 15–25 minutes on 2 vCPUs. Re-running skips finished R2 cells.

In [ ]:
import multiprocessing
NC = multiprocessing.cpu_count()
!python analysis.py --stages fast --nthread {NC}

## 7. Permutation null — the long one

1000 shuffles x 2 arms. Checkpoints every 25, so a disconnect costs at most 25
permutations: **just re-run this cell and it resumes.** The RNG stream is drawn
sequentially regardless of worker count, so the result is identical whether you
run it on 1 core or 8.

In [ ]:
!python analysis.py --stages T1 --nthread 1 --perm-jobs {NC}

## 8. Hyperparameter grid search (optional)

Nested inside the training cohort. Reported as a robustness arm, not the headline —
the permutation null has to run the identical procedure on shuffled labels, and
grid-searching inside 1000 permutations is not feasible, so tuning the observed
value while leaving the null untuned would inflate the p-value.

In [ ]:
!python analysis.py --stages A6 --nthread {NC}

## 9. Build the report and download everything

In [ ]:
!python build_report.py
import json
r = json.load(open('output/results.json'))
print('stages present:', sorted(k for k in r if k not in ('config','versions','data')))
for k, v in r.get('T1', {}).items():
    print(f"  {k}: observed {v['observed']:.4f}  null {v['null_mean']:.3f}"
          f" +/- {v['null_sd']:.3f}  p={v['p_value']:.4f}  ({v['n_perm']} perms,"
          f" complete={v['complete']})")

In [ ]:
from google.colab import files
files.download('output/results.json')
files.download('output/results_report.html')

## 10. Keep results across sessions (recommended)

Colab wipes `/content` when the runtime dies. Mounting Drive and pointing the
output there means a disconnect never costs you permutations.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/alz_results
!cp -n /content/drive/MyDrive/alz_results/results.json output/ 2>/dev/null; true
# ...run your stages here...
!cp output/results.json /content/drive/MyDrive/alz_results/results.json